# Notebook 01 — Options Preprocessing

Handles: Inspection, Cleaning, Filtering, Market Price, DTE, T, Moneyness

**Research Paper:** *Pricing options with a new hybrid neural network model* (Shvimer & Zhu, 2024)

**Input :** `data/raw/underlying_options/UnderlyingOptionsEODCalcs_2023-08-25_cgi_or_historical.csv`  
**Output:** `data/processed/intermediate/options_cleaned.csv`

In [ ]:
import sys, os
# Add src/preprocessing to path so we can import the module
sys.path.insert(0, os.path.join('..', 'src', 'preprocessing'))
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')
print('Libraries loaded.')

## Step 1 — Run the Options Preprocessing Module

This calls `src/preprocessing/options_preprocessing.py` which handles all cleaning and filtering.

In [ ]:
import options_preprocessing

# Run the full pipeline — saves options_cleaned.csv to data/processed/intermediate/
df_opts, quality = options_preprocessing.run(verbose=True)


## Step 2 — Inspect the Cleaned Options Dataset

In [ ]:
print('Shape:', df_opts.shape)
df_opts.head(10)


In [ ]:
print('Column dtypes:')
print(df_opts.dtypes)


In [ ]:
print('Missing values:')
print(df_opts.isnull().sum())


## Step 3 — Column Mapping Reference

| Raw Column | Standardized Name | Notes |
|---|---|---|
| `quote_date` | `date` | Option trading date |
| `expiration` | `expiration` | Expiration date |
| `option_type` (C/P) | `option_type` (CALL/PUT) | Standardized |
| `(underlying_bid_1545 + underlying_ask_1545) / 2` | `S` | Underlying mid-price |
| `strike` | `strike` | Strike price K |
| `bid_1545` | `bid` | Bid price at 15:45 |
| `ask_1545` | `ask` | Ask price at 15:45 |
| `trade_volume` | `volume` | Trading volume |
| `open_interest` | `open_interest` | Open interest |

## Step 4 — Visualise Key Distributions

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(16, 9))
fig.suptitle('Options Cleaned Dataset — Key Distributions', fontsize=14)

df_opts['market_price'].hist(bins=50, ax=axes[0,0])
axes[0,0].set_title('Market Price Distribution')
axes[0,0].set_xlabel('Market Price ($)')

df_opts['days_to_expiration'].hist(bins=40, ax=axes[0,1])
axes[0,1].set_title('Days to Expiration')
axes[0,1].set_xlabel('DTE (days)')

df_opts['moneyness'].hist(bins=50, ax=axes[0,2])
axes[0,2].set_title('Moneyness (S/K)')
axes[0,2].set_xlabel('Moneyness')

df_opts['option_type'].value_counts().plot(kind='bar', ax=axes[1,0])
axes[1,0].set_title('Option Type Count')

df_opts['moneyness_category'].value_counts().plot(kind='bar', ax=axes[1,1])
axes[1,1].set_title('Moneyness Category')

df_opts['S'].hist(bins=40, ax=axes[1,2])
axes[1,2].set_title('Underlying Price S')
axes[1,2].set_xlabel('S ($)')

plt.tight_layout()
plt.savefig(os.path.join('..', 'outputs', 'figures', '01_options_distributions.png'), dpi=100)
plt.show()
print('Plot saved.')

## Step 5 — Data Quality Report

In [ ]:
print('=== OPTIONS DATA QUALITY REPORT ===')
for k, v in quality.items():
    print(f'  {k}: {v}')


## Summary

- Raw dataset: 32,672 rows
- After all filtering (OI, volume, price, DTE, moneyness): **5,829 rows**
- Output saved to `data/processed/intermediate/options_cleaned.csv`
- Ready for VIX and yield curve integration in Notebook 04.